**VISUAL MASK**

In [ ]:
import digitalhub as dh

NOME_PROGETTO = "floods"
project = dh.get_project(NOME_PROGETTO)
print(f"Progetto: {project.name}")

**SETUP PARAMETERS**

In [ ]:
# Parametri Job   
job_name = "anom_complete"                                
                                 
parametri = {     
    "dataset": "Floods_Anomalies_v3_crop_norm",   # artifact con SAR/, OPT/, OPT_RGB/, MASK/ (partNN.zip di .npy)
    "n_samples": 2402,
    "min_flood_pixels": 0,   # considera solo serie con almeno questi pixel classe flood (0 = nessun filtro)
    "flood_class": 2,           # valore della classe flood nella maschera
    "display_gamma": 1.0,     # >1 schiarisce ulteriormente i mezzi toni, solo in visualizzazione
    "job_name": "flood_overlay",
    "save_dir": "/data/flood_overlay",
    "dpi": 100,
}                                                              

# volume -> circa 400 GB dataset Standard
volumi = [
    {
        "volume_type": "ephemeral",
        "name": "volume-spazio-dati",
        "mount_path": "/data",      
        "spec": {"size": "200Gi"}   
    }
]

**BUILD ENVIRONMENT**

In [ ]:
test_train_func = project.new_function(
    name= f'visual_mask_{job_name}',
    kind="python",
    python_version="PYTHON3_10",
    code_src="../src/", 
    handler="utils:visual_mask", 
    base_image="pytorch/pytorch:2.1.2-cuda11.8-cudnn8-runtime",
    requirements=["pandas==2.3.3", "numpy==1.26.4", "rasterio==1.4.4", "tqdm==4.70.0", "tifffile==2024.8.30", "torch==2.1.2", "matplotlib==3.10.9", "digitalhub==0.15.11", "digitalhub-runtime-python==0.15.2", "zarr==2.18.0", "numcodecs==0.13.1", "xarray==2025.6.1"]
)

# .run("build") -> scarica immagine, installa requirements, copia intera cartella code_src, crea immagine docker

build = test_train_func.run("build", wait=True)
print(f"BUILD: {build.status.state}")

**TRAINING**

In [ ]:
# action job = avvia container, esegue script, libera risorse

run_test_encoders = test_train_func.run(
    action="job", 
    parameters=parametri, 
    volumes=volumi, 
    profile="cpu-shared",                                       # 1x = 1 gpu
    # local_execution= True,                                
    wait=True
)

print(f"Run test_encoders avviato: {run_test_encoders.id}")
print(run_test_encoders.status.state)
print(run_test_encoders.status.message)